# Initial Dicom Tags Analysis

The idea here is to cast some light over data validity and data distribution on the dicom tags.
I plan to get insights over 3 perspectives:
  1. All 3 datasets together
  2. Grouping data by dataset
  3. Grouping data by modality

In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import json
from pathlib import Path
from tqdm import tqdm_notebook, tqdm_pandas, tqdm
import math

## Utils

In [10]:
class DataPlotter:
    def __init__(self, data_series):
        if not isinstance(data_series, pd.Series):
            raise ValueError("data_series must be a Pandas Series")
        self.data = data_series

    def plot_bar(self, title="Data Plot", sort=False, ascending=False):
        df = self.data.reset_index()
        df.columns = ["Column", "Value"]
        if sort:
            df = df.sort_values(by="Value", ascending=ascending)

        # Ensure we start a new figure for this plot
        plt.figure(figsize=(10, 6))

        # Create a bar plot on the current figure
        sns.barplot(y="Column", x="Value", data=df)
        plt.title(title)

        # Get and return the current figure
        fig = plt.gcf()
        return fig


class DataframeStatistics:
    def __init__(self, dataframe: pd.DataFrame, title: str = None, color = '', style = 'whitegrid'):
        self.data = dataframe
        self.style = style
        self.title = title
        self.title_font_size = 20
        color_mapping = {'red':'#ce3431',"blue":"#3f5ca7", "green":"#569660", "black": "#292224"}
        self.color = color_mapping['blue']
        if color:
            self.color = color_mapping[color] if color in color_mapping else color

    def compute_summary(self):
        self.unique_values = self.data.nunique()
        self.non_null_count = self.data.count()
        self.nan_count = self.data.isna().sum()
        self.statistics = self.data.describe()
        self.data_types = self.data.dtypes

        # Compute value counts for each column
        self.value_counts_per_column = {col: self.data[col].value_counts() for col in self.data.columns}

        return self.unique_values, self.non_null_count, self.nan_count, self.statistics, self.data_types, self.value_counts_per_column


    def print_column_summary(self):
        if not hasattr(self, 'unique_values'):
            self.compute_summary()
        print("Unique values in each column:\n", self.unique_values)
        print("\nCount of non-null values in each column:\n", self.non_null_count)
        print("\nNumber of NaN values in each column:\n", self.nan_count)
        print("\nBasic statistics for numeric columns:\n", self.statistics)
        #print("\nValue Counts of Columns", self.value_counts_per_column)
        print("\nData types of each column:\n", self.data_types)

    def display_column_summary(self, sort=False, ascending=False):
        if not hasattr(self, 'unique_values'):
            self.compute_summary()
        plotter1 = DataPlotter(self.unique_values)
        plotter1.plot_bar(
            title="Unique Values per Column", sort=sort, ascending=ascending
        ).show()

        plotter2 = DataPlotter(self.non_null_count)
        plotter2.plot_bar(
            title="Non-Null Count per Column", sort=sort, ascending=ascending
        ).show()

        plotter3 = DataPlotter(self.nan_count)
        plotter3.plot_bar(
            title="NaN Count per Column", sort=sort, ascending=ascending
        ).show()

    def generate_plots(self):
        """
        Used for when saving the plots.
        """
        if not hasattr(self, 'unique_values'):
            self.compute_summary()
        yield DataPlotter(self.unique_values).plot_bar(title="Unique Values per Column")
        yield DataPlotter(self.non_null_count).plot_bar(title="Non-Null Count per Column")
        yield DataPlotter(self.nan_count).plot_bar(title="NaN Count per Column")
    
    def generate_data_validation_plots(self, title: str = None, sort=False, ascending=True):
        plt.close('all')  # Close all figures to ensure a fresh start
        sns.set_style(self.style)

        if not hasattr(self, 'unique_values'):
            self.compute_summary()
        
        fig, axs = plt.subplots(2, 2, figsize=(18,15))  # Adjusted to add a row for boxplots
        title_string = title if title else (self.title + ' Data Validity Summary') if self.title else 'Data Validity Summary'
        title_string = title_string+'\n'
        fig.suptitle(title_string, fontsize = self.title_font_size)
        fig.subplots_adjust(top=0.92, bottom=0.08, left=0.10, right=0.95)

        # Bar chart for Unique Values per Column
        if sort:
            self.unique_values = self.unique_values.sort_values(ascending=ascending)
        sns.barplot(x=self.unique_values.values, y=self.unique_values.index, ax=axs[0, 0], color=self.color)
        axs[0, 0].set_title('Unique Values per Column')
        axs[0, 0].set_ylabel('Columns')
        axs[0, 0].set_xlabel('Counts')
        
        # Bar chart for Non-Null Count per Column
        if sort:
            self.non_null_count = self.non_null_count.sort_values(ascending=ascending)
        sns.barplot(x=self.non_null_count.values, y=self.non_null_count.index, ax=axs[0, 1], color=self.color)
        axs[0, 1].set_title('Non-Null Count per Column')
        axs[0, 1].set_ylabel('Columns')
        axs[0, 1].set_xlabel('Counts')
        
        # Bar chart for NaN Count per Column
        if sort:
            self.nan_count = self.nan_count.sort_values(ascending=ascending)
        sns.barplot(x=self.nan_count.values, y=self.nan_count.index, ax=axs[1, 0], color=self.color)
        axs[1, 0].set_title('NaN Count per Column')
        axs[1, 0].set_ylabel('Columns')
        axs[1, 0].set_xlabel('Counts')

        # Bar chart for standard deviation of numeric columns
        std_dev = self.statistics.loc['std'].sort_values(ascending=ascending) if sort else self.statistics.loc['std']
        sns.barplot(x=std_dev.values, y=std_dev.index, ax=axs[1, 1], color=self.color)
        axs[1, 1].set_title('Standard Deviation of Numeric Columns')
        axs[1, 1].set_ylabel('Columns')
        axs[1, 1].set_xlabel('Standard Deviation')

        plt.tight_layout()
        return plt

    def display_data_validation_plots(self, title: str = None, sort=False, ascending=True):
        plot = self.generate_data_validation_plots(sort=sort, ascending=sort, title=title)
        plot.show()
    
    def generate_summary_histplot_plots(self, title: str = None, sort=False, ascending=True):
        plt.close('all')  # Close all figures to ensure a fresh start
        sns.set_style(self.style)

        if not hasattr(self, 'unique_values'):
            self.compute_summary()
        
        fig, axs = plt.subplots(2, 2, figsize=(18,15))  # Adjusted to add a row for boxplots
        title_string = title if title else (self.title + ' Column Data Validity Distribution Summary') if self.title else 'Data Validity Distribution Summary'
        title_string = title_string+'\n'
        fig.suptitle(title_string, fontsize = self.title_font_size)
        fig.subplots_adjust(top=0.92, bottom=0.08, left=0.10, right=0.95)

        # Bar chart for Unique Values per Column
        if sort:
            self.unique_values = self.unique_values.sort_values(ascending=ascending)
        sns.histplot(self.unique_values, ax=axs[0, 0], bins=10, color=self.color)
        axs[0, 0].set_title('Number of Columns with Unique Values per Column')
        axs[0, 0].set_ylabel('Count of Columns')
        axs[0, 0].set_xlabel('Counts')
        
        # Bar chart for Non-Null Count per Column
        if sort:
            self.non_null_count = self.non_null_count.sort_values(ascending=ascending)
        sns.histplot(self.non_null_count, ax=axs[0, 1], bins=10, color=self.color)
        axs[0, 1].set_title('Number of Columns with Non-Null Count per Column')
        axs[0, 1].set_ylabel('Count of Columns')
        axs[0, 1].set_xlabel('Counts')
        
        # Bar chart for NaN Count per Column
        if sort:
            self.nan_count = self.nan_count.sort_values(ascending=ascending)
        sns.histplot(self.nan_count, ax=axs[1, 0], bins=10, color=self.color)
        axs[1, 0].set_title('Number of Columns with NaN Count per Column')
        axs[1, 0].set_ylabel('Count of Columns')
        axs[1, 0].set_xlabel('Counts')

        # Bar chart for standard deviation of numeric columns
        std_dev = self.statistics.loc['std'].sort_values(ascending=ascending) if sort else self.statistics.loc['std']
        sns.histplot(std_dev, ax=axs[1, 1], bins=10, color=self.color)
        axs[1, 1].set_title('Number of Columns with Standard Deviation of Numeric Columns')
        axs[1, 1].set_ylabel('Count of Columns')
        axs[1, 1].set_xlabel('Standard Deviation')

        plt.tight_layout()
        return plt

    def display_summary_histplots_plots(self, title: str = None, sort=False, ascending=True):
        plot = self.generate_summary_histplot_plots(sort=sort, ascending=sort, title=title)
        plot.show()
    
    def generate_boxplots(self, title: str = None):
        plt.close('all')  # Close all figures to ensure a fresh start
        sns.set_style(self.style)


        if not hasattr(self, 'unique_values'):
            self.compute_summary()

        
        numeric_columns = list(self.statistics.columns)  # Numeric columns are in the statistics summary
        n = len(numeric_columns)

        # Remove columns with all NaN values
        numeric_columns = [col for col in numeric_columns if not self.data[col].isna().all()]
        n = len(numeric_columns)  # Update n to reflect the new list length

        dim = int(math.ceil(math.sqrt(n)))
        ncols = nrows = dim
        while ncols * nrows < n:
            nrows += 1
        if ncols * nrows < n:
            ncols += 1

        fig, axs = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
        title_string = title if title else (self.title + ' Boxplots') if self.title else 'Boxplots'
        title_string = title_string+'\n'
        fig.suptitle(title_string, fontsize = self.title_font_size)
        fig.subplots_adjust(top=0.92, bottom=0.08, left=0.10, right=0.95)

        sns.set_style(self.style)

        column_index = 0
        for row in range(nrows):
            for col in range(ncols):
                if column_index >= n:
                    axs[row, col].set_visible(False)  # Hide unused subplot
                    continue

                column = numeric_columns[column_index]
                sns.boxplot(y=self.data[column], ax=axs[row, col], color=self.color)
                axs[row, col].set_title(f'{column}')
                column_index += 1

        plt.tight_layout()
        return plt
    
    def display_boxplots(self, title: str = None):
        plot = self.generate_boxplots(title = title)
        plot.show()
    
    def generate_violin_plots(self, title: str = None):
        plt.close('all')  # Close all figures to ensure a fresh start
        sns.set_style(self.style)

        if not hasattr(self, 'unique_values'):
            self.compute_summary()
        
        numeric_columns = list(self.statistics.columns)  # Numeric columns are in the statistics summary
        n = len(numeric_columns)

        # Remove columns with all NaN values
        numeric_columns = [col for col in numeric_columns if not self.data[col].isna().all()]
        n = len(numeric_columns)  # Update n to reflect the new list length

        dim = int(math.ceil(math.sqrt(n)))
        ncols = nrows = dim
        while ncols * nrows < n:
            nrows += 1
        if ncols * nrows < n:
            ncols += 1

        fig, axs = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
        title_string = title if title else (self.title + ' Violin Plots') if self.title else 'Violin Plots'
        title_string = title_string+'\n'
        fig.suptitle(title_string, fontsize = self.title_font_size)
        fig.subplots_adjust(top=0.92, bottom=0.08, left=0.10, right=0.95)

        sns.set_style(self.style)

        column_index = 0
        for row in range(nrows):
            for col in range(ncols):
                if column_index >= n:
                    axs[row, col].set_visible(False)  # Hide unused subplot
                    continue

                column = numeric_columns[column_index]
                sns.violinplot(y=self.data[column], ax=axs[row, col], color=self.color)
                axs[row, col].set_title(f'{column}')
                column_index += 1

        plt.tight_layout()
        return plt
    
    def display_violin_plots(self, title: str = None):
        plot = self.generate_violin_plots(title = title)
        plot.show()

    def generate_value_counts_charts(self, title: str = None):
        plt.close('all')  # Close all figures to ensure a fresh start
        sns.set_style(self.style)

        if not hasattr(self, 'value_counts_per_column'):
            self.compute_summary()

        categories = list(self.value_counts_per_column.keys())
        n = len(categories)

        # Calculate the layout dimensions
        dim = int(math.ceil(math.sqrt(n)))
        ncols = nrows = dim
        while ncols * nrows < n:
            nrows += 1

        fig, axs = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
        title_string = title if title else (self.title + ' Value Counts Bar Charts') if self.title else 'Value Counts Bar Charts'
        title_string = title_string + '\n'
        fig.suptitle(title_string, fontsize=self.title_font_size)
        fig.subplots_adjust(top=0.92, bottom=0.08, left=0.10, right=0.95)

        sns.set_style(self.style)

        column_index = 0
        for row in range(nrows):
            for col in range(ncols):
                if column_index >= n:
                    axs[row, col].set_visible(False)  # Hide unused subplot
                    continue

                category = categories[column_index]
                counts = self.value_counts_per_column[category]

                # Keep the top 5 values and sum the rest into 'Others'
                top_counts = counts.nlargest(5)
                others = counts.iloc[5:].sum()
                if others > 0:
                    top_counts['Others'] = others

                sns.barplot(x=top_counts.index, y=top_counts, ax=axs[row, col], color=self.color)
                axs[row, col].set_title(f'{category}')
                axs[row, col].tick_params(axis='x', rotation=45)  # Rotate x-axis labels for better readability
                column_index += 1

        plt.tight_layout()
        return plt

    def display_value_counts_charts(self, title: str = None):
        plot = self.generate_value_counts_charts(title = title)
        plot.show()


In [9]:
def clean_dataset_prefix_patient_id(string: str) -> str:
    return string.replace("MIDRC-RICORD-1A-", "").replace("MIDRC-RICORD-1C-", "")


def get_dataset_name_from_patient_id(string: str) -> str:
    if "RICORD" in string:
        return string[:16]
    else:
        return "LIDC-IDRI"


def verbose_drop_na_columns(df):
    original_cols_name = df.shape[1]
    original_cols = df.columns
    df = df.dropna(axis=1, how="all")
    print("Columns dropped:", original_cols_name - df.shape[1])
    print("Columns dropped are : ", set(list(original_cols)) - set(list(df.columns)))
    return df

## Initial csv processing


In [ ]:
CSV_FILEPATH = Path("../preprocess/dicom_tags.csv")
JSON_FILEPATH = Path("../preprocess/dicom_keywords.json")
OUTPUT_PLOTS_FOLDER = Path("../output/paper-figures/preprocess/dataset-analysis")
OUTPUT_PLOTS_FOLDER.mkdir(parents=True, exist_ok=True)

dataset = pd.read_csv(CSV_FILEPATH,delimiter=';')
tag_to_keyword = json.load(open(JSON_FILEPATH))
keyword_to_tag = {keyword: tag for tag, keyword in tag_to_keyword.items()}
display(dataset.head())
print(tag_to_keyword)
print(keyword_to_tag)

_On the json, tags (0013, 0010), (0013, 1010) and (0013, 1013) don't have a keyword mapping, explore this more._

Tags are also not found in any Dicom standard database like Innolitics.

These are private tags, the most relevant one is the (0013, 1010)


In [ ]:
display(dataset[["(0013, 0010)", "(0013, 1010)", "(0013, 1013)"]])

In [ ]:
dataset.columns = [tag_to_keyword[column] for column in dataset.columns]
rearrange_columns = ["Filepath"]
dataset = dataset[
    rearrange_columns + [c for c in dataset.columns if c not in rearrange_columns]
]
display(dataset.head())

print(list(dataset.columns))

Possibly relevant columns:

- PatientID
- PatientName
- Laterality
- Modality
- SmallestImagePixelValue
- LargestImagePixelValue
- BitsStored
- BitsAllocated
- PatientAge
- PatientSex
- ViewPosition
- ImageLaterality
- PatientOrientation
- SpatialResolution
- ImagePosition
- ImageOrientation
- ImagePositionPatient
- BodyPartExamined
- RescaleSlope
- RescaleIntercept
- WindowWidth
- WindowCenter
- Rows
- Columns
- PixelSpacing
- SliceThickness
- PhotometricInterpretation
- SpacingBetweenSlices
- (0013, 1010)


In [ ]:
relevant_columns_list = [
    "PatientID",
    "PatientName",
    "Laterality",
    "Modality",
    "SmallestImagePixelValue",
    "LargestImagePixelValue",
    "BitsStored",
    "BitsAllocated",
    "PatientAge",
    "PatientSex",
    "ViewPosition",
    "ImageLaterality",
    'ImageOrientationPatient',
    "PatientOrientation",
    "SpatialResolution",
    "ImagePosition",
    "ImageOrientation",
    "ImagePositionPatient",
    "BodyPartExamined",
    "RescaleSlope",
    "RescaleIntercept",
    "WindowWidth",
    "WindowCenter",
    "Rows",
    "Columns",
    "PixelSpacing",
    "SliceThickness",
    "PhotometricInterpretation",
    "SpacingBetweenSlices",
    "(0013, 1010)",
    "AcquisitionDate",
    "AcquisitionTime",
    "ImageType",
]
print(relevant_columns_list)
print([keyword_to_tag[keyword] for keyword in relevant_columns_list])

#dataset.to_csv("../output/preprocess/dicom_tags_renamed.csv", sep=";", index=False)
display(dataset[relevant_columns_list].head())

# General Data analysis


In [ ]:
summary = DataframeStatistics(dataset, title='Raw Dataset', color="black")
summary.print_column_summary()
summary.display_data_validation_plots(sort=True)
summary.display_summary_histplots_plots()
plot_obj = summary.generate_summary_histplot_plots()
plot_obj.savefig(OUTPUT_PLOTS_FOLDER/'raw-dataset_summary_distribution.png')
plot_obj.savefig(OUTPUT_PLOTS_FOLDER/'raw-dataset_summary_distribution.svg')
summary.display_boxplots()

Insights:

- Most dicoms are 16 bits, but some are 8 bit
- Every dicom had the Filepath successfully stored (no null values)
-


## Initial Analysis of the Relevant Columns

In [ ]:
#print(type(dataset['ImageOrientationPatient'][0]))
#print(dataset['ImageOrientationPatient'].dtype)

#groups = dataset['ImageOrientationPatient'].value_counts()
#print(groups)

#dataset['ImageOrientationPatient'] = dataset['ImageOrientationPatient'].apply(lambda x: str(x).replace(',','/'))
#print(type(dataset['ImageOrientationPatient'][0]))
#groups = dataset['ImageOrientationPatient'].value_counts()
#print(groups)
#groups = dataset['ViewPosition'].value_counts()
#print(groups)
#display(dataset[relevant_columns_list]['ImageOrientationPatient'])

summary = DataframeStatistics(dataset[relevant_columns_list], title="Relevant Columns", color="black")
summary.print_column_summary()
summary.display_data_validation_plots(sort=True)
plot_obj = summary.generate_data_validation_plots(sort=True)
plot_obj.savefig(OUTPUT_PLOTS_FOLDER/'raw-dataset-relevant_summary.png')
plot_obj.savefig(OUTPUT_PLOTS_FOLDER/'raw-dataset-relevant_summary.svg')
summary.display_summary_histplots_plots()
plot_obj = summary.generate_summary_histplot_plots()
plot_obj.savefig(OUTPUT_PLOTS_FOLDER/'raw-dataset-relevant_summary_distribution.png')
plot_obj.savefig(OUTPUT_PLOTS_FOLDER/'raw-dataset-relevant_summary_distribution.svg')
summary.display_boxplots()
summary.display_violin_plots()
summary.display_value_counts_charts()


In [ ]:
#DataframeStatistics(dataset[relevant_columns_list]).display_column_summary(sort=True)

In [ ]:
display(dataset.head())
dataset = verbose_drop_na_columns(dataset)
display(dataset.head())

In [ ]:
summary = DataframeStatistics(dataset, title='Dataset (NA Columns Dropped)', color="black")
summary.print_column_summary()
summary.display_data_validation_plots(sort=True)
summary.display_summary_histplots_plots()
plot_obj = summary.generate_summary_histplot_plots()
plot_obj.savefig(OUTPUT_PLOTS_FOLDER/'raw-dataset_summary_distribution-NA-Dropped.png')
plot_obj.savefig(OUTPUT_PLOTS_FOLDER/'raw-dataset_summary_distribution-NA-Dropped.svg')
summary.display_boxplots()

# Per Modality Analysis


In [ ]:
ct_dataset = dataset[dataset["Modality"] == "CT"]
xray_dataset = dataset[dataset["Modality"] != "CT"]
display(ct_dataset.head())
display(xray_dataset.head())

Still many NaN columns due to the fact CTs don't store XRayTubeCurrent, for example, and all columns are initialized using NaN.


In [ ]:
ct_relevant_columns_list = [column for column in relevant_columns_list if column in list(ct_dataset.columns)]
summary = DataframeStatistics(ct_dataset[ct_relevant_columns_list], title='CT Relevant Columns', color="blue")
summary.print_column_summary()
summary.display_data_validation_plots(sort=True)
summary.display_summary_histplots_plots()
summary.display_boxplots()
summary.display_violin_plots()

In [ ]:
xray_relevant_columns_list = [column for column in relevant_columns_list if column in list(xray_dataset.columns)]
summary = DataframeStatistics(xray_dataset[xray_relevant_columns_list], title='X-Rays Relevant Columns', color="red")
summary.print_column_summary()
summary.display_data_validation_plots(sort=True)
summary.display_summary_histplots_plots()
summary.display_boxplots()
summary.display_violin_plots()

## Initial data cleaning


In [ ]:
ct_dataset_cleaned = verbose_drop_na_columns(ct_dataset)
xray_dataset_cleaned = verbose_drop_na_columns(xray_dataset)
display(ct_dataset_cleaned.head())

display(xray_dataset_cleaned.head())

In [ ]:
summary = DataframeStatistics(ct_dataset_cleaned, title='CT Dataset with NA Columns Dropped', color="blue")
summary.print_column_summary()
summary.display_data_validation_plots(sort=True)
summary.display_summary_histplots_plots()
summary.display_boxplots()

In [ ]:
summary = DataframeStatistics(xray_dataset_cleaned, title='X-Rays Dataset with NA Columns Dropped', color="red")
summary.print_column_summary()
summary.display_data_validation_plots(sort=True)
summary.display_summary_histplots_plots()
summary.display_boxplots()

In [ ]:
# Using the cleaned x-ray dataset
cleaned_xray_dataset_relevant_columns = [column for column in relevant_columns_list if column in list(xray_dataset_cleaned.columns)]

# Using the cleaned CT dataset
cleaned_ct_dataset_relevant_columns = [column for column in relevant_columns_list if column in list(ct_dataset_cleaned.columns)]


In [ ]:
summary = DataframeStatistics(xray_dataset_cleaned[cleaned_xray_dataset_relevant_columns], title='X-Rays Dataset Relevant Columns (NA Columns Dropped)', color="red")
summary.print_column_summary()
summary.display_data_validation_plots(sort=True)
summary.display_summary_histplots_plots()
summary.display_boxplots()
summary.display_violin_plots()
summary.display_value_counts_charts()
summary.display_value_counts_charts()

In [ ]:
summary = DataframeStatistics(ct_dataset_cleaned[cleaned_ct_dataset_relevant_columns], title='CTs Dataset Relevant Columns (NA Columns Dropped)', color="blue")
summary.print_column_summary()
summary.display_data_validation_plots(sort=True)
summary.display_summary_histplots_plots()
summary.display_boxplots()
summary.display_violin_plots()
summary.display_value_counts_charts()

Insights:

 - SliceThickness might be useful due to most of the distribution be around 0.6-2.5
 - 